# Protein Sequence Tokenization Stretegy Comparative Analysis

This notebook implements the phase-2 Tokenization of animo acids and compare strategies based on Vocab size and

<strong>_out of scope:_</strong>

- compare strategies based on Generative and Classification tasks


In [1]:
import os
import sys
import gc
import json
from pathlib import Path

from collections import Counter
from itertools import chain

import math

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

from IPython.display import display
from tqdm.auto import tqdm

from util.file_utils import ensure_directories, iter_dataset
from util.text_similarity import rank_texts
from util.seq_util import inspect_residue_distribution
from tokenizer_module import create_tokenizer

In [2]:
REPO_ROOT = Path.cwd().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data/interim/protein_eda/protein_corpus.parquet"
OUTPUT_DIR = REPO_ROOT / "data/interim/protein_tokenization"
FIGURE_OUTPUT_DIR = REPO_ROOT / "results/protein_tokenization/figures"
ARTIFACT_OUTPUT_DIR = REPO_ROOT / "results/protein_tokenization/artifacts"

MAX_SEQ_LENGTH = 512
PARSE_BATCH_SIZE = 2**16
# TOKENIZATION_STRATEGIES = ["Unigram", "BPE", "WordPiece", "words", "pairs", "k-mers"]
TOKENIZATION_STRATEGIES = ["BPE"]
MIN_FREQUENCIES = [3]
KMER_SIZES = [3, 5, 6, 7, 9]
VOCAB_SIZES = [256, 512]
RARE_RESIDUE_POLICY = "keep"  #  None replace_with_unk replace_with_nn
SAMPLE_MODE = None  # None  #500

SAVE_ARTIFACTS = True
SAVE_FIGURES = True

TOKENIZER_CONFIGS = (
    [{"name": "words", "requires_training": False}]
    + [{"name": "Unigram", "requires_training": True, "vocab_size": vs} for vs in VOCAB_SIZES]
    + [{"name": "BPE", "requires_training": True, "vocab_size": vs, "min_frequency": mf} for vs in VOCAB_SIZES for mf in MIN_FREQUENCIES]
    + [{"name": "WordPiece", "requires_training": True, "vocab_size": vs} for vs in VOCAB_SIZES]
    + [{"name": "pairs", "requires_training": False, "pair_mode": "sliding"}]
    + [{"name": "k-mers", "requires_training": False, "k": k, "mode": "sliding"} for k in KMER_SIZES]
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
ensure_directories(ARTIFACT_OUTPUT_DIR, FIGURE_OUTPUT_DIR, OUTPUT_DIR)


## Parse Corpus

Parse every available `protein.faa` file, preserve provenance, and surface malformed records or headers.


In [3]:
protein_dataset = ds.dataset(str(DATA_PATH), format="parquet")

dataset_schema = pd.DataFrame(
    {
        "column": protein_dataset.schema.names,
        "dtype": [str(t) for t in protein_dataset.schema.types],
    }
)
records_total_count = protein_dataset.count_rows()
print(f"Records: {records_total_count:,}")

Records: 88,497,376


In [4]:
if SAMPLE_MODE is not None:
    print(f"Taking a sample of {SAMPLE_MODE} records")
    protein_dataset = ds.dataset(protein_dataset.head(SAMPLE_MODE))
    records_total_count = protein_dataset.count_rows()


display(dataset_schema)

,column,dtype
0,assembly_id,string
1,sequence_id,string
2,ambiguous_residues,string
3,functional_annotations,string
4,raw_header,string
5,sequence_length,int64
6,sequence,string


In [5]:
TOTAL_RECORDS = records_total_count
TOTAL_BATCHES = math.ceil(TOTAL_RECORDS / PARSE_BATCH_SIZE)
print(f"Total batches (batch size {PARSE_BATCH_SIZE}): {TOTAL_BATCHES:,}")

Total batches (batch size 65536): 1,351


In [6]:
# # find all unique functional_annotations and count their occurrences
# annotation_counts = Counter()

# for batch in iter_dataset(protein_dataset, batch_size=PARSE_BATCH_SIZE, desc="Counting annotations"):
#     df_batch = batch.to_pandas()
#     annotation_counts.update(df_batch["functional_annotations"].explode())

# annotation_counts_df = pd.DataFrame(annotation_counts.items(), columns=["annotation", "count"]).sort_values(by="count", ascending=False)
# del annotation_counts
# del df_batch
# del batch
# gc.collect()


In [7]:
# if SAVE_ARTIFACTS:
#     annotation_counts_df.to_csv(ARTIFACT_OUTPUT_DIR / "annotation_counts.csv", index=False)

# print(f"Number of Annotations: {annotation_counts_df.shape[0]:_}")
# display(annotation_counts_df.sort_values(by="annotation", ascending=True).head(10))

In [8]:
# annotation_counts_df.head(10).plot(kind="bar", x="annotation", y="count", title="Functional Annotation Counts", width=1.0)
# plt.xticks(rotation=45, ha="right")
# plt.tight_layout()

### Annotation Similarity Search

Use the reusable text ranking utility to search functional annotations with either fuzzy matching or semantic similarity.


In [9]:
# semantic_annotation_matches = rank_texts(annotation_counts_df["annotation"], "anti microbial resistance", method="semantic", threshold=0.25)
# del annotation_counts_df
# gc.collect()

# display(semantic_annotation_matches.top_k(40))

In [10]:
# if SAVE_ARTIFACTS:
#     semantic_annotation_matches.top_k(40).reset_index(drop=True).to_csv(ARTIFACT_OUTPUT_DIR / "semantic_annotation_matches.csv", index=False)

# del semantic_annotation_matches
# gc.collect()

#### MLM vs next-token prediction

MLM

- sees left + right context

- good for representation learning

Causal LM

- only sees left context

- better for generation

For proteins, MLM is often a very strong choice unless you specifically want sequence generation.


In [11]:
# STANDARD_RESIDUES = set("ACDEFGHIKLMNPQRSTVWY")
# RARE_RESIDUES = set("XBZJUO")  # misstyped tokens that appear in the dataset
# VALID_RESIDUES = STANDARD_RESIDUES | RARE_RESIDUES

# SPECIAL_TOKENS = [
#     "[PAD]",  # padding
#     "[UNK]",  # unknown token for out-of-vocabulary residues
#     "[CLS]",  # start of sequence (classification token)
#     "[SEP]",  # separator token (not used in single sequence tasks but reserved for potential future use)
#     "[MASK]",  # mask token for masked language modeling
#     "[UNKAA]",  # optional custom placeholder token for rare residues (if using replace_with_unk policy)
# ]

check correct resudue and distribution


In [12]:
# AA = np.frombuffer(b"ACDEFGHIKLMNPQRSTVWYXBZUOJ", dtype=np.uint8)
# aa_set = set(AA.tolist())
# residue_totals = np.zeros(256, dtype=np.int64)

# for batch in iter_dataset(protein_dataset, batch_size=PARSE_BATCH_SIZE, desc="Counting residues"):
#     seqs = batch.column("sequence").to_pylist()
#     blob = "".join(seqs).encode("ascii", errors="ignore")
#     arr = np.frombuffer(blob, dtype=np.uint8)
#     residue_totals += np.bincount(arr, minlength=256)

# rows = [(chr(i), int(residue_totals[i])) for i in sorted(aa_set) if residue_totals[i] > 0]
# residue_counts = Counter(dict(rows))
# df = pd.DataFrame(residue_counts.items(), columns=["residue", "count"]).sort_values("residue").reset_index(drop=True)
# display(df)

In [13]:
# if SAVE_ARTIFACTS:
#     df.to_csv(ARTIFACT_OUTPUT_DIR / "residue_counts.csv", index=False)

# del residue_counts
# del df
# del batch
# gc.collect()

In [14]:
# def normalize_sequence(seq, rare_residue_policy="keep"):
#     """
#     Normalize one protein sequence.
#     - uppercases
#     - strips spaces
#     - handles rare/unknown residues
#     - drops invalid chars
#     """
#     seq = seq.upper().replace(" ", "")

#     out = []
#     for ch in seq:
#         if ch in STANDARD_RESIDUES:
#             out.append(ch)
#         elif ch in RARE_RESIDUES:
#             if rare_residue_policy == "keep":
#                 out.append(ch)
#             elif rare_residue_policy == "replace_with_unk":
#                 out.append("[UNKAA]")  # optional custom placeholder token
#             elif rare_residue_policy == "replace_with_nn":
#                 out.append("X")
#             else:
#                 raise ValueError(f"Unknown rare_residue_policy: {rare_residue_policy}")
#         else:
#             # silently drop junk chars for now
#             continue

#     # join carefully because [UNKAA] is multi-char
#     # later we will space-split tokens, so convert sequence into token list first
#     return " ".join(out)

In [15]:
# def iter_bpe_training_corpus(protein_table, batch_size=PARSE_BATCH_SIZE, rare_residue_policy="keep"):
#     """
#     Yield normalized, space-separated sequences for tokenizer training.

#     Example output:
#         'M K T F F V'
#         'A C D E'
#     """

#     for batch in tqdm(protein_table.to_batches(max_chunksize=batch_size), total=TOTAL_BATCHES, desc="Preparing BPE corpus"):
#         seq_col = batch.column("sequence")
#         for seq_scalar in seq_col:
#             if not seq_scalar.is_valid:
#                 continue

#             seq = seq_scalar.as_py()
#             if not seq:
#                 continue

#             norm = normalize_sequence(seq, rare_residue_policy=rare_residue_policy)
#             if norm:
#                 yield norm

In [16]:
# def train_bpe_tokenizer(protein_table, save_dir, vocab_size, min_frequency, batch_size=PARSE_BATCH_SIZE, rare_residue_policy="keep"):
#     """
#     Train a Hugging Face BPE tokenizer on protein sequences.
#     """
#     save_dir = Path(save_dir)
#     save_dir.mkdir(parents=True, exist_ok=True)

#     # BPE model
#     tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

#     # Since our input is already 'M K T F ...', split on whitespace
#     tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

#     trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=min_frequency, special_tokens=SPECIAL_TOKENS)

#     tokenizer.train_from_iterator(
#         iterator=iter_bpe_training_corpus(
#             protein_table=protein_table,
#             batch_size=batch_size,
#             rare_residue_policy=rare_residue_policy,
#         ),
#         trainer=trainer,
#     )

#     # Add BERT-style post-processing
#     cls_id = tokenizer.token_to_id("[CLS]")
#     sep_id = tokenizer.token_to_id("[SEP]")

#     tokenizer.post_processor = processors.TemplateProcessing(
#         single="[CLS] $A [SEP]",
#         pair="[CLS] $A [SEP] $B:1 [SEP]:1",
#         special_tokens=[
#             ("[CLS]", cls_id),
#             ("[SEP]", sep_id),
#         ],
#     )

#     tokenizer.enable_padding(
#         pad_id=tokenizer.token_to_id("[PAD]"),
#         pad_token="[PAD]",
#         length=MAX_SEQ_LENGTH,
#     )

#     tokenizer.enable_truncation(max_length=MAX_SEQ_LENGTH)

#     tokenizer_path = save_dir / "tokenizer.json"
#     tokenizer.save(str(tokenizer_path))

#     print(f"Saved BPE tokenizer to: {tokenizer_path}")
#     print(f"Vocab size learned: {tokenizer.get_vocab_size()}")

#     return tokenizer

In [17]:
# def preview_tokenizer(tokenizer: Tokenizer, sequences: list[str], rare_residue_policy: str = "keep") -> None:
#     """
#     Quick preview of tokenization behavior.
#     """
#     for seq in sequences:
#         norm = normalize_sequence(seq, rare_residue_policy=rare_residue_policy)
#         enc = tokenizer.encode(norm)
#         print("=" * 80)
#         print("RAW:   ", seq)
#         print("NORM:  ", norm)
#         print("TOKENS:", enc.tokens)
#         print("IDS:   ", enc.ids)

In [18]:
# def view_vocab(tokenizer, n=20, sort_by="id"):
#     """
#     # View tokenizer vocab.

#     sort_by:
#         - "id"   → default order (merge order-ish)
#         - "alpha"→ alphabetical
#         - "len"  → token length
#     """
#     vocab = tokenizer.get_vocab()  # dict: token -> id

#     if sort_by == "id":
#         items = sorted(vocab.items(), key=lambda x: x[1])
#     elif sort_by == "alpha":
#         items = sorted(vocab.items(), key=lambda x: x[0])
#     elif sort_by == "len":
#         items = sorted(vocab.items(), key=lambda x: (len(x[0]), x[0]))
#     else:
#         raise ValueError("sort_by must be one of: id, alpha, len")

#     print(f"Showing top {n} tokens (sorted by {sort_by}):\n")
#     for token, idx in items[:n]:
#         print(f"{idx:>4}  {token}")


# def count_learned_tokens(tokenizer):
#     vocab = tokenizer.get_vocab()
#     learned = [t for t in vocab if len(t) > 1 and not t.startswith("[")]
#     print(f"Learned tokens: {len(learned)}")

In [19]:
# def tokenizer_pipeline(tokenizer, sequences):
#     for seq in sequences:
#         norm = normalize_sequence(seq)
#         yield tokenizer.encode(norm)


In [ ]:
for strategy in TOKENIZATION_STRATEGIES:
    tokenizer_configs = [conf for conf in TOKENIZER_CONFIGS if conf["name"] == strategy]
    # residue_counts = inspect_residue_distribution(protein_dataset, batch_size=PARSE_BATCH_SIZE)

    print(f"Processing strategy: {strategy}")
    for config in tokenizer_configs:
        name = config["name"]
        save_dir = ARTIFACT_OUTPUT_DIR / f"tokenizers/{name}/vocab{config.get('vocab_size', 'default')}_{RARE_RESIDUE_POLICY}"

        tok = create_tokenizer(
            max_seq_length=MAX_SEQ_LENGTH,
            rare_residue_policy=RARE_RESIDUE_POLICY,
            **config,
        )

        if config["requires_training"]:
            tok.train(
                protein_table=protein_dataset,
                save_dir=save_dir,
                batch_size=PARSE_BATCH_SIZE,
                total_batches=TOTAL_BATCHES,
                **{k: v for k, v in config.items() if k not in ("name", "requires_training", "pair_mode")},
            )
        elif name in ("k-mers", "pairs"):
            tok.build_vocab(protein_dataset, batch_size=PARSE_BATCH_SIZE, total_batches=TOTAL_BATCHES)
            tok.save(save_dir)

        tok.count_learned_tokens()
        # tok.view_vocab(n=40)
        # tok.preview(protein_dataset.head(3)["sequence"].to_pylist())

Processing strategy: BPE


Preparing BPE corpus:   0%|          | 0.00/88.5M [00:00<?, ?rows/s]

In [ ]:
# tok.view_vocab(n=40)
tok.preview(protein_dataset.head(3)["sequence"].to_pylist())

RAW:    MKRISTTITTTITITTGNGAG
NORM:   MKRISTTITTTITITTGNGAG
TOKENS: ['[CLS]', 'MK', 'RI', 'ST', 'TI', 'TT', 'TI', 'TI', 'TT', 'GN', 'GAG', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]